# 🔌 Adding Custom Distribution in Climatology Engine

This notebook demonstrates how to add a new statistical distribution to the system.

**What you will learn:**
- Distribution plugin architecture
- `DistributionPlugin` class structure
- Implementing `fit()` method for a new distribution
- Adding Weibull distribution as an example
- Registering the new distribution in the system
- Testing the new distribution on sample data
- Comparing the new distribution with existing ones

---

## 📐 Distribution Plugin Architecture

The system uses a plugin architecture. Each new distribution must:

1. Be placed in `plugins/distributions/` folder
2. Inherit from `DistributionPlugin` class
3. Implement the `fit()` method
4. Have a name, code, parameters, and parameter count

### Base Class Structure

```python
class DistributionPlugin:
    name = None          # Distribution name
    code = None          # Unique integer code
    params = []          # Parameter names
    n_params = 0         # Number of parameters
    
    def fit(self, data):
        # Implement fitting
        return {'p1': val1, 'p2': val2, 'loglik': ..., 'aicc': ..., 'bic': ...}
```

---

In [ ]:
import sys
import os
project_root = os.path.abspath('..')
if project_root not in sys.path:
    sys.path.insert(0, project_root)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from scipy import stats
from core.engine.distribution_plugin import DistributionPlugin
from core.engine.plugin_loader import load_plugins

sns.set_style('whitegrid')
plt.rcParams['font.size'] = 11
print('✅ Libraries loaded.')

In [ ]:
# ============================================================================
# 1. Define Custom Distribution: Weibull
# ============================================================================

class WeibullDistribution(DistributionPlugin):
    """
    Weibull Distribution
    f(x) = (shape/scale) * (x/scale)^(shape-1) * exp(-(x/scale)^shape)
    """
    name = "Weibull"
    code = 5  # New code (after 0-4)
    params = ["shape", "scale"]
    n_params = 2
    supports_negative = False
    supports_zero = True
    supports_positive = True
    extreme_only = False

    def fit(self, data):
        """
        Fit Weibull distribution using Maximum Likelihood Estimation
        """
        data = np.array(data)[~np.isnan(data)]
        if len(data) < 3:
            return {
                'shape': np.nan,
                'scale': np.nan,
                'loglik': np.nan,
                'aicc': np.nan,
                'bic': np.nan
            }
        
        if np.any(data <= 0):
            data = data - np.min(data) + 1e-6
        
        try:
            shape, loc, scale = stats.weibull_min.fit(data, floc=0)
            
            loglik = np.sum(stats.weibull_min.logpdf(data, shape, loc=0, scale=scale))
            
            n = len(data)
            k = self.n_params
            aicc = -2*loglik + 2*k + (2*k*(k+1))/(n-k-1) if n > k+1 else np.inf
            bic = -2*loglik + k*np.log(n)
            
            return {
                'shape': shape,
                'scale': scale,
                'loglik': loglik,
                'aicc': aicc,
                'bic': bic
            }
        except Exception as e:
            return {
                'shape': np.nan,
                'scale': np.nan,
                'loglik': np.nan,
                'aicc': np.nan,
                'bic': np.nan
            }

    def pdf(self, x, params):
        """Probability density function"""
        shape = params.get('shape', 1.0)
        scale = params.get('scale', 1.0)
        return stats.weibull_min.pdf(x, shape, loc=0, scale=scale)

print("✅ Weibull distribution defined.")
print(f"   Name: {WeibullDistribution.name}")
print(f"   Code: {WeibullDistribution.code}")
print(f"   Parameters: {WeibullDistribution.params}")

In [ ]:
# ============================================================================
# 2. Test Weibull Distribution on Synthetic Data
# ============================================================================

np.random.seed(42)
shape_true = 2.0
scale_true = 10.0
synthetic_data = stats.weibull_min.rvs(shape_true, loc=0, scale=scale_true, size=500)

print(f"📊 Synthetic Weibull data:")
print(f"   True Shape: {shape_true}")
print(f"   True Scale: {scale_true}")
print(f"   Data size: {len(synthetic_data)}")

weibull_dist = WeibullDistribution()
result = weibull_dist.fit(synthetic_data)

print("\n📈 Fit results:")
for key, value in result.items():
    if isinstance(value, float):
        print(f"   {key}: {value:.4f}")
    else:
        print(f"   {key}: {value}")

In [ ]:
# Plot histogram and Weibull curve
fig, ax = plt.subplots(figsize=(10, 6))

ax.hist(synthetic_data, bins=40, density=True, alpha=0.5, color='blue', 
        edgecolor='black', label='Data')

x = np.linspace(0, max(synthetic_data) * 1.1, 500)
params = {'shape': result['shape'], 'scale': result['scale']}
pdf = weibull_dist.pdf(x, params)
ax.plot(x, pdf, 'r-', linewidth=2.5, label=f'Weibull(shape={result["shape"]:.2f}, scale={result["scale"]:.2f})')

ax.axvline(shape_true, color='green', linestyle='--', linewidth=2, label=f'True Shape: {shape_true}')
ax.axvline(scale_true, color='orange', linestyle='--', linewidth=2, label=f'True Scale: {scale_true}')

ax.set_xlabel('Value', fontsize=12)
ax.set_ylabel('Probability Density', fontsize=12)
ax.set_title('Weibull Distribution Fit on Synthetic Data', fontsize=14, fontweight='bold')
ax.legend(loc='upper right', fontsize=10)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# ============================================================================
# 3. Load Existing Plugins
# ============================================================================

existing_plugins = load_plugins()
print(f"✅ Existing distributions: {len(existing_plugins)}")
for code, dist in existing_plugins.items():
    print(f"   [{code}] {dist.name}")

all_plugins = existing_plugins.copy()
all_plugins[WeibullDistribution.code] = WeibullDistribution()
distributions = {dist.name: dist for dist in all_plugins.values()}

print(f"\n✅ Total distributions after adding Weibull: {len(distributions)}")
print(f"   Distributions: {list(distributions.keys())}")

In [ ]:
# ============================================================================
# 4. Test New Distribution on Sample Data
# ============================================================================

sample_dir = os.path.join(project_root, 'sample_data')
station_files = sorted([f for f in os.listdir(sample_dir) if f.endswith('.csv')])
station_data = pd.read_csv(os.path.join(sample_dir, station_files[0]))
data = station_data.values
data_year = data[:365, 1]  # tmean

data_positive = data_year - np.min(data_year) + 0.1

weibull_result = WeibullDistribution().fit(data_positive)

print("📈 Weibull fit results on sample temperature data:")
for key, value in weibull_result.items():
    if isinstance(value, float):
        print(f"   {key}: {value:.4f}")
    else:
        print(f"   {key}: {value}")

In [ ]:
# ============================================================================
# 5. Compare Weibull with Other Distributions
# ============================================================================

all_results = {}
print("\n🔄 Fitting all distributions (including Weibull)...")

for name, dist in distributions.items():
    try:
        if name == 'Weibull':
            res = dist.fit(data_positive)
        else:
            res = dist.fit(data_year)
        all_results[name] = res
        print(f"✅ {name}: AICc = {res.get('aicc', np.nan):.2f}")
    except Exception as e:
        print(f"❌ {name}: Error - {str(e)}")
        all_results[name] = None

In [ ]:
# Select the best model
valid_results = {k: v for k, v in all_results.items() 
                  if v is not None and 'aicc' in v and not np.isnan(v['aicc'])}

if valid_results:
    best_name = min(valid_results, key=lambda x: valid_results[x]['aicc'])
    print("=" * 60)
    print(f"🏆 Best distribution: {best_name}")
    print(f"   AICc: {valid_results[best_name]['aicc']:.4f}")
    print("=" * 60)

comparison_df = pd.DataFrame([{
    'Distribution': name,
    'AICc': res['aicc'],
    'N_Params': res.get('n_params', np.nan)
} for name, res in valid_results.items()])
comparison_df = comparison_df.sort_values('AICc').reset_index(drop=True)
comparison_df.index = comparison_df.index + 1
comparison_df

In [ ]:
# Plot AICc comparison with Weibull
fig, ax = plt.subplots(figsize=(10, 6))

colors = ['#2ecc71' if name == best_name else '#e74c3c' if name == 'Weibull' else '#3498db' 
          for name in comparison_df['Distribution']]

bars = ax.barh(comparison_df['Distribution'], comparison_df['AICc'], 
               color=colors, alpha=0.7, edgecolor='black', linewidth=1)

ax.set_xlabel('AICc', fontsize=12)
ax.set_title('AICc Comparison with Weibull Distribution', fontsize=14, fontweight='bold')
ax.grid(True, alpha=0.3, axis='x')

for bar, val in zip(bars, comparison_df['AICc']):
    ax.text(val + 0.5, bar.get_y() + bar.get_height()/2, f'{val:.1f}', 
            va='center', fontsize=10, fontweight='bold')

plt.tight_layout()
plt.show()

In [ ]:
# ============================================================================
# 6. Quality Evaluation for Weibull
# ============================================================================

from core.quality.quality_flag import QualityFlag

flags = QualityFlag.evaluate(weibull_result, data_positive)

flag_names = {
    QualityFlag.PASS: '✅ PASS',
    QualityFlag.LOW_SAMPLE: '⚠️ LOW_SAMPLE',
    QualityFlag.NO_CONVERGENCE: '❌ NO_CONVERGENCE',
    QualityFlag.OUTLIER: '⚠️ OUTLIER',
    QualityFlag.HIGH_AICC: '⚠️ HIGH_AICC',
    QualityFlag.BAD_SKEW: '⚠️ BAD_SKEW',
    QualityFlag.NAN_INPUT: '❌ NAN_INPUT',
    QualityFlag.INF_INPUT: '❌ INF_INPUT',
}

print("📋 Weibull fit quality:")
for flag in flags:
    print(f"   {flag_names.get(flag, f'UNKNOWN ({flag})')}")

In [ ]:
# ============================================================================
# 7. Save New Distribution as Plugin File
# ============================================================================

plugin_code = '''#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
plugins/distributions/weibull.py
Weibull Distribution
"""

import numpy as np
from scipy import stats
from core.engine.distribution_plugin import DistributionPlugin

class WeibullDistribution(DistributionPlugin):
    name = "Weibull"
    code = 5
    params = ["shape", "scale"]
    n_params = 2
    supports_negative = False
    supports_zero = True
    supports_positive = True
    extreme_only = False

    def fit(self, data):
        data = np.array(data)[~np.isnan(data)]
        if len(data) < 3:
            return {
                'shape': np.nan,
                'scale': np.nan,
                'loglik': np.nan,
                'aicc': np.nan,
                'bic': np.nan
            }
        if np.any(data <= 0):
            data = data - np.min(data) + 1e-6
        try:
            shape, loc, scale = stats.weibull_min.fit(data, floc=0)
            loglik = np.sum(stats.weibull_min.logpdf(data, shape, loc=0, scale=scale))
            n = len(data)
            k = self.n_params
            aicc = -2*loglik + 2*k + (2*k*(k+1))/(n-k-1) if n > k+1 else np.inf
            bic = -2*loglik + k*np.log(n)
            return {
                'shape': shape,
                'scale': scale,
                'loglik': loglik,
                'aicc': aicc,
                'bic': bic
            }
        except Exception as e:
            return {
                'shape': np.nan,
                'scale': np.nan,
                'loglik': np.nan,
                'aicc': np.nan,
                'bic': np.nan
            }

    def pdf(self, x, params):
        shape = params.get('shape', 1.0)
        scale = params.get('scale', 1.0)
        return stats.weibull_min.pdf(x, shape, loc=0, scale=scale)
'''

plugin_dir = os.path.join(project_root, 'plugins', 'distributions')
os.makedirs(plugin_dir, exist_ok=True)

plugin_path = os.path.join(plugin_dir, 'weibull.py')
with open(plugin_path, 'w', encoding='utf-8') as f:
    f.write(plugin_code)

print(f"✅ Weibull plugin saved to {plugin_path}")
print("\n📌 To use Weibull distribution in the project:")
print("   1. The file weibull.py is in plugins/distributions/")
print("   2. It will be auto-discovered by load_plugins()")
print("   3. You can use WeibullDistribution directly.")

## 📋 Summary

In this notebook you learned:

✅ Distribution plugin architecture in Climatology Engine
✅ `DistributionPlugin` class structure
✅ Implementing `fit()` method for Weibull distribution
✅ Adding a new distribution to the system
✅ Testing new distribution on synthetic and real data
✅ Comparing new distribution with existing ones
✅ Quality evaluation of the new distribution
✅ Saving new distribution as a plugin file

---

**Key Takeaways:**

1. Each new distribution must inherit from `DistributionPlugin`.
2. The `fit()` method must return a dict with parameters, `loglik`, `aicc`, and `bic`.
3. The distribution code must be unique (from 5 onward).
4. New distributions go in `plugins/distributions/` folder.
5. The system auto-discovers plugins automatically.

---

**Next Steps:**
- Notebook 10: Advanced Usage